In [0]:
from datetime import datetime
import uuid

run_id = "uc16_" + datetime.utcnow().strftime("%Y%m%d_%H%M%S")
task_name = "build_appointments_clean"

spark.sql(f"""
INSERT INTO poc_health_no_show.ops.pipeline_run_audit
VALUES (
  '{run_id}',
  'uc16_no_show_daily_pipeline',
  '{task_name}',
  'STARTED',
  current_timestamp(),
  NULL,
  NULL,
  NULL
)
""")

In [0]:
%sql
CREATE OR REPLACE TABLE poc_health_no_show.silver.appointments_clean
COMMENT 'Cleaned and standardized appointment records. Date-level validation is used because AppointmentDay is date-only in the source dataset.'
AS
WITH ranked AS (
  SELECT
    *,
    row_number() OVER (
      PARTITION BY appointment_id
      ORDER BY _ingested_at DESC
    ) AS rn
  FROM poc_health_no_show.bronze.appointments_raw
),
typed AS (
  SELECT
    appointment_id,
    sha2(patient_id, 256) AS patient_pseudo_id,
    CASE
      WHEN upper(trim(gender)) IN ('M', 'MALE') THEN 'M'
      WHEN upper(trim(gender)) IN ('F', 'FEMALE') THEN 'F'
      ELSE 'UNKNOWN'
    END AS gender,
    TRY_CAST(scheduled_day AS TIMESTAMP) AS scheduled_ts,
    TRY_CAST(appointment_day AS TIMESTAMP) AS appointment_ts,
    CAST(TRY_CAST(scheduled_day AS TIMESTAMP) AS DATE) AS scheduled_date,
    CAST(TRY_CAST(appointment_day AS TIMESTAMP) AS DATE) AS appointment_date,
    TRY_CAST(age AS INT) AS age_raw,
    initcap(trim(neighbourhood)) AS neighbourhood,
    CAST(TRY_CAST(scholarship AS INT) = 1 AS BOOLEAN) AS scholarship_flag,
    CAST(TRY_CAST(hipertension AS INT) = 1 AS BOOLEAN) AS hypertension_flag,
    CAST(TRY_CAST(diabetes AS INT) = 1 AS BOOLEAN) AS diabetes_flag,
    CAST(TRY_CAST(alcoholism AS INT) = 1 AS BOOLEAN) AS alcoholism_flag,
    CAST(TRY_CAST(handcap AS INT) > 0 AS BOOLEAN) AS handicap_flag,
    CAST(TRY_CAST(sms_received AS INT) = 1 AS BOOLEAN) AS sms_received_flag,
    CAST(upper(trim(no_show)) = 'YES' AS BOOLEAN) AS no_show_flag,
    _ingested_at
  FROM ranked
  WHERE rn = 1
)
SELECT
  appointment_id,
  patient_pseudo_id,
  gender,
  scheduled_ts,
  appointment_ts,
  scheduled_date,
  appointment_date,
  age_raw AS age,
  neighbourhood,
  scholarship_flag,
  hypertension_flag,
  diabetes_flag,
  alcoholism_flag,
  handicap_flag,
  sms_received_flag,
  no_show_flag,
  datediff(appointment_date, scheduled_date) AS lead_time_days,
  date_format(appointment_date, 'EEEE') AS appointment_weekday,
  CASE
    WHEN appointment_id IS NULL THEN 'MISSING_APPOINTMENT_ID'
    WHEN age_raw IS NULL THEN 'INVALID_AGE'
    WHEN age_raw < 0 OR age_raw > 120 THEN 'INVALID_AGE'
    WHEN scheduled_ts IS NULL THEN 'INVALID_SCHEDULED_DATE'
    WHEN appointment_ts IS NULL THEN 'INVALID_APPOINTMENT_DATE'
    WHEN scheduled_date > appointment_date THEN 'SCHEDULED_AFTER_APPOINTMENT'
    ELSE 'VALID'
  END AS data_quality_status,
  current_timestamp() AS _processed_at
FROM typed;

In [0]:
%sql
SELECT
  data_quality_status,
  COUNT(*) AS appointment_count
FROM poc_health_no_show.silver.appointments_clean
GROUP BY data_quality_status
ORDER BY appointment_count DESC;

In [0]:
%sql
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT appointment_id) AS distinct_appointments,
  SUM(CASE WHEN no_show_flag THEN 1 ELSE 0 END) AS no_show_count,
  ROUND(
    SUM(CASE WHEN no_show_flag THEN 1 ELSE 0 END) / COUNT(*) * 100,
    2
  ) AS no_show_rate_pct
FROM poc_health_no_show.silver.appointments_clean;

In [0]:
spark.sql(f"""
INSERT INTO poc_health_no_show.ops.pipeline_run_audit
VALUES (
  '{run_id}',
  'uc16_no_show_daily_pipeline',
  '{task_name}',
  'SUCCESS',
  current_timestamp(),
  current_timestamp(),
  NULL,
  NULL
)
""")

In [0]:
%pip install dlt

In [0]:
%restart_python

In [0]:
import dlt as dp
from pyspark.sql import functions as F

# ---------------------------------------------------------
# Bronze: raw appointment data
# ---------------------------------------------------------
@dp.table(
    name="appointments_bronze",
    comment="Raw appointment records ingested from source."
)
def appointments_bronze():
    return (
        spark.read.table("poc_health_no_show.bronze.appointments_raw")
    )


# ---------------------------------------------------------
# Silver: cleaned appointment data with expectations
# ---------------------------------------------------------
@dp.table(
    name="appointments_clean",
    comment="Cleaned appointment records with data-quality expectations."
)
@dp.expect("valid_appointment_id", "appointment_id IS NOT NULL")
@dp.expect("valid_age", "age IS NULL OR (age >= 0 AND age <= 120)")
@dp.expect(
    "valid_date_sequence",
    "scheduled_date IS NULL OR appointment_date IS NULL OR scheduled_date <= appointment_date"
)
def appointments_clean():
    return (
        dp.read("appointments_bronze")
        .withColumn("appointment_id", F.trim(F.col("appointment_id")))
        .withColumn("age", F.col("age").cast("int"))
        .withColumn("scheduled_date", F.to_date("scheduled_date"))
        .withColumn("appointment_date", F.to_date("appointment_date"))
    )


# ---------------------------------------------------------
# Gold: valid appointments only
# ---------------------------------------------------------
@dp.table(
    name="appointments_gold",
    comment="Valid appointment records for reporting and analytics."
)
def appointments_gold():
    return (
        dp.read("poc_health_no_show.silver.appointments_clean")
        .filter(
            F.col("appointment_id").isNotNull()
            & (F.col("age").isNull() | ((F.col("age") >= 0) & (F.col("age") <= 120)))
            & (
                F.col("scheduled_date").isNull()
                | F.col("appointment_date").isNull()
                | (F.col("scheduled_date") <= F.col("appointment_date"))
            )
        )
    )